# Aufgabe 1 · UFO-Akten: Was zählen wir eigentlich?

[Aufgabenstellung](README.md) · [Walkthrough](WALKTHROUGH.md) · [Aktenleseführer](../../docs/AKTENLESEFUEHRER.md)

Die Forschungsredaktion bereitet eine Geschichte über die Western-USA-Akten vor. Drei Dokumente schildern Wahrnehmungen und bewerten einen Teil davon. Kann die Redaktion daraus die Anzahl beobachteter Ereignisse ableiten? Welche Zahlen kann Dein Katalog tatsächlich belegen?

Du baust eine relationale Sicht, prüfst ihre Integrität und lieferst einen Quellenkatalog mit sauber benannten Zähleinheiten. **Teil A und B bilden die Aufgabe; Teil C ist eine Vertiefung.**

Trage Deine Lösungen in die mit TODO markierten Zellen ein. Unbearbeitete Abschnitte melden OFFEN und werden übersprungen; ein fehlerfreier erster Durchlauf bedeutet noch nicht, dass die Aufgabe gelöst ist.

## 0. Eigenständiger Einstieg

Kernel **Python (rothstein-storage-workshop-2026)**; Zellen von oben nach unten ausführen. Du brauchst die Kursumgebung und die mitgelieferten CSV-Dateien, keine vorherige Demo-Datenbank. SQLite benötigt keinen Server und kein Docker.

`data/input/relational/` und die sieben Original-PDFs bleiben unverändert. Arbeitsdaten liegen unter `data/work/uap/`.

In [1]:
from pathlib import Path
import sys
import sqlite3
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/sqlite_workshop.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Öffne das Notebook innerhalb des entpackten Repositories.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from sqlite_workshop import connect, create_schema, load_snapshot, read_snapshot, query, table_counts
pd.set_option("display.max_colwidth", 90)
print("SQLite:", sqlite3.sqlite_version)

DB_PATH = ROOT / "data/work/uap/storage.sqlite"
ready = False
completed = set()
inputs = read_snapshot("uap")
print("Arbeitsdatenbank:", DB_PATH)
display(pd.DataFrame({"Tabelle": inputs.keys(), "Zeilen": [len(x) for x in inputs.values()]}))
display(pd.DataFrame(inputs["catalog_entries"])[["entry_id", "source_key", "title"]].head(3))

SQLite: 3.53.4
Arbeitsdatenbank (relativ zum Repo): data/work/uap/storage.sqlite


,Tabelle,Zeilen
0,agencies,10
1,releases,5
2,catalog_entries,375
3,assets,374
4,entry_assets,397
5,portal_pairings,336


,entry_id,source_key,title
0,entry_003dca02286e780af10f,DOW-UAP-PR123,"DOW-UAP-PR123, Unresolved UAP Report, Pacific Ocean, 2019"
1,entry_00a8b83d2d5e598e801e,NASA-UAP-D025,"NASA-UAP-D025, “Apollo 16 Scientific Debriefing”"
2,entry_0177dbb4d82c0471a1d9,DOW-UAP-PR122,"DOW-UAP-PR122, Unresolved UAP Report, Gulf of Oman, 2021"


## A1 · Schlüssel und Beziehungen

Lies das [Basisschema](../../schemas/sqlite/uap_base.sql). Die Tabellen `agencies`, `releases`, `catalog_entries`, `assets` und `portal_pairings` sind vorbereitet. `release_date` wird nur in `releases` gespeichert; der Import prüft vorher die Übereinstimmung mit der redundanten Spalte der CSV.

Ergänze `entry_assets` mit:

- `entry_id` und `asset_id`: nicht leer; Referenzen auf ihre jeweiligen Stammtabellen;
- `source_field`: nicht leer; Herkunft des Verweises;
- einem Schlüssel, der **dieselbe Eintrag–Asset-Kombination** nicht zweimal zulässt;
- `STRICT` wie im Basisschema.

Eine Tabelle ohne Schlüsselbeziehungen genügt nicht. Verwende `CREATE TABLE IF NOT EXISTS`, damit ein zweiter Durchlauf funktioniert.

**Begründe:** Warum ist `source_key` kein geeigneter Primärschlüssel? Warum genügt in `entry_assets` keine der beiden IDs allein?

In [2]:
# TODO: Ersetze None durch Dein CREATE-TABLE-Statement als mehrzeiligen String.
# Strukturhilfe: CREATE TABLE IF NOT EXISTS entry_assets (...) STRICT;
bridge_ddl = None

### Deine Modellentscheidung

1. Primärschlüssel und erhaltene Quellinformation: …
2. Bedeutung einer Zeile in `entry_assets`: …
3. Warum eine n:m-Beziehung? …

## A2 · Laden, erneut laden und Integrität prüfen

Der vorbereitete Import ersetzt die Inhalte der sechs UAP-Tabellen **gemeinsam in einer Transaktion**. Bei einem Fehler bleiben die vorher bestätigten Daten erhalten. Bei Erfolg stellt er den Snapshot wieder her; eigene Änderungen in diesen Tabellen werden verworfen.

`CREATE TABLE IF NOT EXISTS` ändert keine vorhandene Definition. Wenn Du das Schema nach einem ersten Versuch korrigierst: Schliesse die Verbindungen, benenne **nur Deine Arbeitsdatenbank** um und führe das Notebook erneut aus. Die Musterlösung besitzt eine andere Datei. Details stehen in [README.md](README.md).

In [3]:
if bridge_ddl is None:
    print("OFFEN: entry_assets in A1 definieren.")
else:
    with connect(DB_PATH) as con:
        create_schema(con, "uap", bridge_ddl)
        first = load_snapshot(con, "uap")
    with connect(DB_PATH) as con:
        persisted = table_counts(con, "uap")
        second = load_snapshot(con, "uap")
    assert first == persisted == second
    assert second == {"agencies":10, "releases":5, "catalog_entries":375,
                      "assets":374, "entry_assets":397, "portal_pairings":336}
    ready = True
    completed.add("Import")
    print("IMPORT OK: neu geöffnet und wiederholt geladen, keine Verdoppelung.")

OFFEN: entry_assets in A1 definieren.


### Vorhersage vor dem Versuch

Die nächste Zelle schreibt zunächst eine technische Probe-Stelle und danach eine ungültige Zuordnung. Sage voraus, was bei jedem Versuch passieren muss. Der Code rollt **die gesamte Probe** zurück; auch der erste Schreibzugriff darf danach nicht mehr sichtbar sein.

Das zeigt Atomarität und die Einhaltung definierter Integritätsregeln. Es prüft weder die Wahrheit einer Zeugenaussage noch alle ACID-Eigenschaften unter Hardwareausfällen.

In [4]:
# Alle Probeänderungen werden zurückgerollt, auch bei einem falschen Schema.
# Vorhersage: Welcher Constraint muss den zweiten Schreibzugriff ablehnen?
if ready:
    from uuid import uuid4
    with connect(DB_PATH) as con:
        entry, asset = con.execute("SELECT entry_id, asset_id FROM entry_assets LIMIT 1").fetchone()
        probes = {
            "fehlender Katalogeintrag": ("_missing_" + uuid4().hex, asset, "probe"),
            "fehlendes Asset": (entry, "_missing_" + uuid4().hex, "probe"),
            "doppelte Zuordnung": (entry, asset, "probe"),
        }
        for name, values in probes.items():
            marker = "_probe_" + uuid4().hex
            rejected = False
            con.execute("BEGIN")
            try:
                con.execute("INSERT INTO agencies VALUES (?, ?)", (marker, marker))
                con.execute("INSERT INTO entry_assets VALUES (?, ?, ?)", values)
            except sqlite3.IntegrityError as error:
                rejected = True
                print(name, "abgelehnt:", error)
            finally:
                con.execute("ROLLBACK")
            assert rejected, f"Constraint fehlt: {name}"
            assert con.execute("SELECT COUNT(*) FROM agencies WHERE agency_id=?", (marker,)).fetchone()[0] == 0
        assert con.execute("PRAGMA foreign_key_check").fetchall() == []
    print("INTEGRITÄT OK: beide Fremdschlüssel, zusammengesetzter Schlüssel und Rollback geprüft.")
else:
    print("OFFEN: zuerst Teil A ergänzen und importieren.")

if ready:
    completed.add("Integrität")

OFFEN: zuerst Teil A ergänzen und importieren.


In [5]:
if ready:
    display(query(DB_PATH, """
        SELECT e.entry_id, e.source_key, e.title, r.release_date
        FROM catalog_entries e JOIN releases r ON r.release_id=e.release_id
        WHERE e.source_key=:key ORDER BY r.release_date, e.entry_id
    """, {"key": "FBI-UAP-D014"}))

## B1 · Drei Akten für die Redaktion zusammenstellen

Liefere pro ausgewähltem Katalogeintrag eine Zeile mit `entry_id`, `source_key`, `title`, `agency`, `release_date` und `asset_references`.
Verbinde Katalog, Stelle und Veröffentlichung. Ergänze die Zahl verknüpfter Assets, ohne Einträge ohne Verweis durch einen Inner Join auszuschliessen. Gruppiere nach dem **technischen Eintrag**, nicht nur nach dem Aktenkürzel.

Die Werte werden als Parameter übergeben. Hinweise: `JOIN`, `LEFT JOIN`, `COUNT(ea.asset_id)`, `GROUP BY` und `IN (:witness1, :witness2, :analysis)`.

In [6]:
case_parameters = {"witness1":"DOW-UAP-D079", "witness2":"DOW-UAP-D080", "analysis":"DOW-UAP-D077"}
# TODO: Ergänze Deine parametrisierte SELECT-Abfrage.
sql_dossier = None
if ready and sql_dossier:
    dossier = query(DB_PATH, sql_dossier, case_parameters)
    display(dossier)
    assert set(dossier["source_key"]) == set(case_parameters.values()) and len(dossier) == 3
    assert dossier["entry_id"].nunique() == 3
    assert dossier["asset_references"].tolist() == [1, 1, 1]
    assert set(dossier["agency"]) == {"Department of War"}
    assert set(dossier["release_date"]) == {"2026-06-12"}
    assert dossier["title"].notna().all()
    completed.add("Quellenkatalog")
else:
    print("OFFEN: Import und B1 ergänzen.")

OFFEN: Import und B1 ergänzen.


### Die Zahlen mit den Originalen verbinden

Öffne [D079, Seiten 1–2](../../data/raw/originals/DOW-UAP-D079.pdf), [D080, besonders Seite 5](../../data/raw/originals/DOW-UAP-D080.pdf) und [D077, Seiten 1–4](../../data/raw/originals/DOW-UAP-D077.pdf).

- Was ist eine Zeugenschilderung, was eine nachträgliche Illustration und was eine Analyse?
- Auf welchen Teil der Schilderungen bezieht sich die Analyse?
- Weshalb folgt aus drei Einträgen und drei verlinkten PDFs nicht die Zahl realer Sichtungen oder unabhängiger Bestätigungen?

Halte Deine Begründung mit Aktenkürzel und PDF-Seite fest. Der [Aktenleseführer](../../docs/AKTENLESEFUEHRER.md) bietet Orientierung; die Originale sind englisch.

**Deine Einordnung mit Seitenbelegen:** …

## B2 · Den ganzen Bestand korrekt zählen

Die Redaktion fragt: «Sind das 375 Fälle, 397 Dateien oder 374 Ereignisse?» Korrigiere die Einheiten und liefere **eine SQL-Ergebniszeile** mit:

- `catalog_entries`: verschiedene technische Katalogeinträge;
- `entry_asset_links`: Eintrag–Asset-Zuordnungen;
- `distinct_asset_references`: verschiedene referenzierte Asset-IDs.

Starte bei `catalog_entries` und verwende einen `LEFT JOIN` auf `entry_assets`. Wo brauchst Du `DISTINCT`, wo würde es die falsche Einheit zählen? Ein Asset bezeichnet im Lehrmodell einen normalisierten Datei-/Medienverweis. Identische Dateiinhalte hinter verschiedenen URLs sind damit nicht nachgewiesen.

In [7]:
# TODO: Ergänze die Bestandszählung mit den drei vereinbarten Spaltennamen.
sql_counts = None
if ready and sql_counts:
    counts = query(DB_PATH, sql_counts)
    display(counts)
    expected = {"catalog_entries":375, "entry_asset_links":397, "distinct_asset_references":374}
    assert len(counts) == 1 and counts.iloc[0].to_dict() == expected
    completed.add("Bestandszählung")
else:
    print("OFFEN: Import und B2 ergänzen.")

OFFEN: Import und B2 ergänzen.


### Dein Veröffentlichungstext

Formuliere zwei Sätze für die Redaktion: einen mit den Zähleinheiten und einen mit der Grenze der Aussage. Ergänze anschliessend Deine Modellentscheidung: Welche Abfrage ist relational gut ausdrückbar, welche Inhaltsstruktur oder Pfadsuche wäre in einem anderen Modell bequemer?

**Dein Veröffentlichungstext:** …

**Deine Modellentscheidung:** …

## C · Vertiefung: Stellenvergleich und Index

Aggregiere dieselben drei Kennzahlen zusätzlich je Katalogstelle. Prüfe insbesondere `Department of War`. Weshalb ist seine Zahl verknüpfter Assets kleiner als seine Zahl von Katalogeinträgen? Welche Bedeutung hat `COUNT(*)` nach dem Join?

Ergänze danach einen Index auf `catalog_entries(source_key)`. Untersuche die Suche nach `FBI-UAP-D014` mit `EXPLAIN QUERY PLAN`. Ein nicht eindeutiger Index unterstützt den Zugriff und erhält beide Einträge. Ein `UNIQUE`-Index wäre hier fachlich falsch.

Aus einem Plan auf diesem kleinen Bestand folgt keine allgemeine Leistungsaussage. [SQLite: Abfrageplan](https://www.sqlite.org/eqp.html).

In [8]:
# Optional: dieselben Zähleinheiten pro Stelle.
sql_by_agency = None
if ready and sql_by_agency:
    by_agency = query(DB_PATH, sql_by_agency)
    display(by_agency)
    dow = by_agency.set_index("agency").loc["Department of War"]
    assert dow.to_dict() == {"catalog_entries":190, "entry_asset_links":211, "distinct_asset_references":189}
else:
    print("VERTIEFUNG: Abfrage pro Stelle noch nicht ergänzt.")

VERTIEFUNG: Abfrage pro Stelle noch nicht ergänzt.


In [9]:
lookup_sql = """SELECT entry_id, source_key, title
                FROM catalog_entries WHERE source_key=:key"""
lookup_parameters = {"key": "FBI-UAP-D014"}
# Optional: Ergänze den nicht eindeutigen Index.
index_ddl = None
if ready and index_ddl:
    print("Plan vor diesem CREATE INDEX (bei erneutem Lauf kann der Index bereits existieren):")
    display(query(DB_PATH, "EXPLAIN QUERY PLAN " + lookup_sql, lookup_parameters))
    with connect(DB_PATH) as con:
        con.execute(index_ddl)
    print("Plan danach:")
    display(query(DB_PATH, "EXPLAIN QUERY PLAN " + lookup_sql, lookup_parameters))
    display(query(DB_PATH, lookup_sql, lookup_parameters))
else:
    print("VERTIEFUNG: Index noch nicht ergänzt.")

VERTIEFUNG: Index noch nicht ergänzt.


## Abschluss

Prüfe die technischen Rückmeldungen und Deine schriftlichen Begründungen. Die Checks vergleichen Referenzwerte und einige Integritätsregeln; sie ersetzen keine fachliche Prüfung Deiner Interpretation.

In [10]:
required = {"Import", "Integrität", "Quellenkatalog", "Bestandszählung"}
missing = sorted(required - completed)
if missing:
    print("AUFGABE OFFEN:", ", ".join(missing))
else:
    print("SQLITE AUFGABE TECHNISCH OK. Schriftliche Begründungen und Seitenbelege gemeinsam besprechen.")

AUFGABE OFFEN: Bestandszählung, Import, Integrität, Quellenkatalog
